# Python & NumPy Engineering

**TL;DR:** Beyond CV, they want clean, **fault-tolerant** Python: vectorized NumPy
(no per-pixel Python loops), a batch processor that survives bad files, and basic
algorithmic thinking. The JD literally says "fault-tolerant… manage resources in
long-running processes" — show that.

In [ ]:
%matplotlib inline
import cv2, numpy as np, matplotlib.pyplot as plt, os
from collections import deque

def show(img, title=""):
    """Display a BGR or grayscale image inline."""
    if img is None:
        print("image is None"); return
    plt.figure(figsize=(5,5))
    if img.ndim == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis("off"); plt.show()

def make_sample_assets():
    """Create synthetic inputs so every cell runs without your own files.
    Swap input.jpg for a real photo any time to experiment."""
    doc = np.full((700,500,3), 30, np.uint8)
    quad = np.array([[120,90],[420,140],[400,560],[90,520]], np.int32)
    cv2.fillConvexPoly(doc, quad, (235,235,235))
    cv2.putText(doc, "LABEL 12345", (150,300), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (20,20,20), 2)
    cv2.imwrite("input.jpg", doc)
    cv2.imwrite("noisy.jpg", doc)
    cv2.imwrite("low.jpg", (doc*0.3 + 90).astype(np.uint8))
    os.makedirs("ind", exist_ok=True)
    cv2.imwrite("ind/a.png", doc); cv2.imwrite("ind/b.png", doc)
    open("ind/c.png","wb").write(b"corrupt-not-an-image")
    vw = cv2.VideoWriter("test.avi", cv2.VideoWriter_fourcc(*"MJPG"), 10, (320,240))
    for i in range(12):
        fr = np.zeros((240,320,3), np.uint8)
        cv2.rectangle(fr, (10+i*15,80), (50+i*15,140), (255,255,255), -1)
        vw.write(fr)
    vw.release()
    return doc

doc = make_sample_assets()
print("Sample assets ready: input.jpg, noisy.jpg, low.jpg, ind/, test.avi")
show(doc, "synthetic input.jpg  (replace with your own image!)")


> **Tip:** run the setup cell above first. Each problem below defines its solution and then runs a quick demo. Edit and re-run any cell to experiment.

## Problem 16 — Vectorize it (no pixel loops)

**Tests:** that you reach for NumPy, not `for y: for x:`. Example: threshold +
tint without loops.

In [ ]:
import numpy as np
import cv2

def red_tint_bright_areas(path, out="p16_out.png", thr=180):
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > thr                       # boolean array, fully vectorized
    img[mask] = [0, 0, 255]                 # set all bright pixels red (BGR)
    cv2.imwrite(out, img)
    return img

# Per-channel mean without loops:
def channel_means(img):
    return img.reshape(-1, 3).mean(axis=0)  # (B_mean, G_mean, R_mean)

In [ ]:
show(red_tint_bright_areas("input.jpg"), "Problem 16 - bright areas tinted")

In [ ]:
print("BGR means:", channel_means(cv2.imread("input.jpg")))

**Watch out:** boolean-mask indexing (`img[mask] = ...`) and `axis=` reductions are
the whole point. If you wrote a double `for` loop over pixels, that's the wrong
answer — say "vectorize with NumPy."

---

## Problem 17 — Fault-tolerant batch image processor

**Tests:** the "manage a long-running job, don't crash on one bad input" skill.
Process a folder, skip and log corrupt files, keep going.

In [ ]:
import os
import cv2
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

def process_folder(in_dir, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    exts = (".jpg", ".jpeg", ".png", ".bmp")
    ok = fail = 0
    for name in sorted(os.listdir(in_dir)):
        if not name.lower().endswith(exts):
            continue
        src = os.path.join(in_dir, name)
        try:
            img = cv2.imread(src)
            if img is None:
                raise ValueError("unreadable / corrupt")
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            cv2.imwrite(os.path.join(out_dir, name), gray)
            ok += 1
        except Exception as e:                      # one bad file can't kill the run
            logging.warning("skipped %s: %s", name, e)
            fail += 1
    logging.info("done: %d ok, %d failed", ok, fail)
    return ok, fail

In [ ]:
print("(ok, failed) =", process_folder("ind","outd"))

**Watch out:** `cv2.imread` returns **None** on failure (it doesn't raise) — guard
it. Catch per-file so the batch continues; log what you skipped. That *is* fault
tolerance.

---

## Problem 18 — Count connected components (blobs)

**Tests:** a small algorithm framed in CV terms; thresholding + labeling.

In [ ]:
import cv2

def count_blobs(path, min_area=50):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    _, th = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(th)
    # label 0 is the background; filter tiny specks by area
    blobs = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= min_area]
    return len(blobs), centroids[blobs]

In [ ]:
n, cent = count_blobs("input.jpg"); print("blobs:", n)

If asked to do it **without** OpenCV (pure algorithm), flood-fill / BFS over the
binary grid:

In [ ]:
from collections import deque

def count_islands(grid):
    """grid: 2D list of 0/1. Counts 4-connected components of 1s."""
    if not grid:
        return 0
    R, C = len(grid), len(grid[0])
    seen = [[False] * C for _ in range(R)]
    count = 0
    for r in range(R):
        for c in range(C):
            if grid[r][c] == 1 and not seen[r][c]:
                count += 1
                q = deque([(r, c)]); seen[r][c] = True
                while q:
                    y, x = q.popleft()
                    for dy, dx in ((1,0),(-1,0),(0,1),(0,-1)):
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < R and 0 <= nx < C and grid[ny][nx] == 1 \
                           and not seen[ny][nx]:
                            seen[ny][nx] = True; q.append((ny, nx))
    return count

In [ ]:
print("islands:", count_islands([[1,1,0,0],[0,1,0,1],[0,0,0,1]]))

**Watch out:** `connectedComponentsWithStats` labels background as 0 — start your
loop at 1. The pure-Python flood fill is the classic "number of islands" if they
want a DSA flavor.